In [1]:
import json
from pathlib import Path

In [2]:
OCR_PATH = Path("./data/tesseract/devis_2.json")
LABEL_PATH = Path("./data/label_studio/label_studio_export.json")

with OCR_PATH.open("r", encoding="utf-8") as f:
    ocr_data = json.load(f)

with LABEL_PATH.open("r", encoding="utf-8") as f:
    annotations = json.load(f)

In [3]:
print(ocr_data["document"])
print(ocr_data["page_count"])
print(ocr_data["pages"][0]["word_count"])

devis_2.pdf
299
1


In [4]:
print(annotations[0].keys())
print(annotations[0]["annotations"][0].keys())
print(annotations[0]["annotations"][0]["result"])

dict_keys(['id', 'annotations', 'file_upload', 'drafts', 'predictions', 'data', 'meta', 'created_at', 'updated_at', 'allow_skip', 'inner_id', 'total_annotations', 'cancelled_annotations', 'total_predictions', 'comment_count', 'unresolved_comment_count', 'last_comment_updated_at', 'project', 'updated_by', 'comment_authors'])
dict_keys(['id', 'completed_by', 'result', 'was_cancelled', 'ground_truth', 'created_at', 'updated_at', 'draft_created_at', 'lead_time', 'prediction', 'result_count', 'unique_id', 'import_id', 'last_action', 'bulk_created', 'task', 'project', 'updated_by', 'parent_prediction', 'parent_annotation', 'last_created_by'])
[{'original_width': 1654, 'original_height': 2339, 'image_rotation': 0, 'value': {'x': 76.06765123551924, 'y': 1.340616274042484, 'width': 12.112241988221072, 'height': 1.787488365389979, 'rotation': 0, 'rectanglelabels': ['DATE_DEVIS']}, 'id': 'Ew5npKGHnV', 'from_name': 'label', 'to_name': 'image', 'type': 'rectanglelabels', 'origin': 'manual'}, {'orig

In [5]:
def parse_label_studio_annotations(annotations: list[dict]) -> list[dict]:
    results = annotations[0]["annotations"][0]["result"]

    parsed_annotations = []

    for result in results:
        if result["type"] != "rectanglelabels":
            continue

        value = result["value"]

        x1 = round(value["x"] * 10)
        y1 = round(value["y"] * 10)
        x2 = round((value["x"] + value["width"]) * 10)
        y2 = round((value["y"] + value["height"]) * 10)

        parsed_annotations.append(
            {
                "label": value["rectanglelabels"][0],
                "bbox": [x1, y1, x2, y2],
            }
        )

    return parsed_annotations

In [6]:
entity_regions = parse_label_studio_annotations(annotations)

for region in entity_regions:
    print(region)

{'label': 'DATE_DEVIS', 'bbox': [761, 13, 882, 31]}
{'label': 'NUMERO_DEVIS', 'bbox': [256, 150, 382, 172]}
{'label': 'FOURNISSEUR', 'bbox': [36, 6, 289, 113]}
{'label': 'SOCIETE', 'bbox': [691, 57, 772, 75]}
{'label': 'MONTANT_TOTAL', 'bbox': [851, 751, 950, 768]}


In [7]:
OCR_IMAGE_WIDTH = 1191
OCR_IMAGE_HEIGHT = 1684

In [8]:
def normalize_bbox(
    bbox: list[int],
    image_width: int,
    image_height: int,
) -> list[int]:
    x1, y1, x2, y2 = bbox

    return [
        round(1000 * x1 / image_width),
        round(1000 * y1 / image_height),
        round(1000 * x2 / image_width),
        round(1000 * y2 / image_height),
    ]

In [9]:
ocr_words = []

for word in ocr_data["pages"][0]["words"]:
    ocr_words.append(
        {
            "text": word["text"],
            "confidence": word["confidence"],
            "bbox": normalize_bbox(
                word["bbox"],
                OCR_IMAGE_WIDTH,
                OCR_IMAGE_HEIGHT,
            ),
        }
    )

In [10]:
for word in ocr_words[:10]:
    print(word)

{'text': 'Sud-Ouest', 'confidence': 0.79, 'bbox': [113, 69, 196, 87]}
{'text': 'ie', 'confidence': 0.49, 'bbox': [90, 85, 168, 109]}
{'text': 'tige', 'confidence': 0.41, 'bbox': [175, 85, 208, 109]}
{'text': 'Offre', 'confidence': 0.96, 'bbox': [62, 156, 119, 171]}
{'text': 'de', 'confidence': 0.96, 'bbox': [128, 156, 149, 169]}
{'text': 'prix', 'confidence': 0.96, 'bbox': [157, 156, 207, 172]}
{'text': 'n°:', 'confidence': 0.71, 'bbox': [215, 157, 251, 169]}
{'text': '002.143/25', 'confidence': 0.85, 'bbox': [259, 157, 373, 169]}
{'text': 'Affaire', 'confidence': 0.87, 'bbox': [59, 177, 104, 187]}
{'text': ':', 'confidence': 0.87, 'bbox': [109, 180, 113, 185]}


In [11]:
def center(bbox: list[int]) -> tuple[int, int]:
    """
    Retourne le centre d'une bounding box.

    bbox = [x1, y1, x2, y2]
    """

    x1, y1, x2, y2 = bbox

    center_x = round((x1 + x2) / 2)
    center_y = round((y1 + y2) / 2)

    return center_x, center_y

In [12]:
bbox = [113, 69, 196, 87]

print(center(bbox))

(154, 78)


In [13]:
def point_in_bbox(
    point: tuple[int, int],
    bbox: list[int],
) -> bool:
    """
    Vérifie si un point est contenu
    dans une bounding box.
    """

    px, py = point

    x1, y1, x2, y2 = bbox

    return (
        x1 <= px <= x2
        and
        y1 <= py <= y2
    )

In [14]:
bbox = [100, 50, 200, 100]

print(point_in_bbox((150, 70), bbox))
print(point_in_bbox((250, 70), bbox))

True
False


In [15]:
def assign_entity_label(
    word_bbox: list[int],
    entity_regions: list[dict],
) -> str:
    """
    Retourne le label métier
    correspondant à un mot OCR.

    Si aucun rectangle ne contient
    le mot, retourne "O".
    """

    word_center = center(word_bbox)

    for entity in entity_regions:

        if point_in_bbox(
            word_center,
            entity["bbox"],
        ):
            return entity["label"]

    return "O"

In [16]:
for word in ocr_words[:20]:

    label = assign_entity_label(
        word["bbox"],
        entity_regions,
    )

    print(
        f'{word["text"]:20} -> {label}'
    )

Sud-Ouest            -> FOURNISSEUR
ie                   -> FOURNISSEUR
tige                 -> FOURNISSEUR
Offre                -> O
de                   -> O
prix                 -> O
n°:                  -> O
002.143/25           -> NUMERO_DEVIS
Affaire              -> O
:                    -> O
A24013               -> O
CHEVAL               -> O
Devis                -> O
effectué             -> O
par                  -> O
Olivier              -> O
CANTELAUBE           -> O
Beautiran,           -> O
le                   -> DATE_DEVIS
28                   -> DATE_DEVIS


In [17]:
labeled_words = []

for word in ocr_words:
    entity_label = assign_entity_label(
        word["bbox"],
        entity_regions,
    )

    labeled_words.append(
        {
            **word,
            "entity_label": entity_label,
        }
    )

In [18]:
for word in labeled_words[:20]:
    print(
        f'{word["text"]:20} -> {word["entity_label"]}'
    )

Sud-Ouest            -> FOURNISSEUR
ie                   -> FOURNISSEUR
tige                 -> FOURNISSEUR
Offre                -> O
de                   -> O
prix                 -> O
n°:                  -> O
002.143/25           -> NUMERO_DEVIS
Affaire              -> O
:                    -> O
A24013               -> O
CHEVAL               -> O
Devis                -> O
effectué             -> O
par                  -> O
Olivier              -> O
CANTELAUBE           -> O
Beautiran,           -> O
le                   -> DATE_DEVIS
28                   -> DATE_DEVIS


In [19]:
def convert_to_bio_labels(entity_labels: list[str]) -> list[str]:
    bio_labels = []
    previous_label = "O"

    for current_label in entity_labels:
        if current_label == "O":
            bio_labels.append("O")
            previous_label = "O"
            continue

        if current_label != previous_label:
            bio_labels.append(f"B-{current_label}")
        else:
            bio_labels.append(f"I-{current_label}")

        previous_label = current_label

    return bio_labels

In [20]:
entity_labels = [
    assign_entity_label(
        word["bbox"],
        entity_regions,
    )
    for word in ocr_words
]

In [21]:
bio_labels = convert_to_bio_labels(entity_labels)

In [22]:
for word, entity_label, bio_label in zip(
    ocr_words[:30],
    entity_labels[:30],
    bio_labels[:30],
):
    print(
        f'{word["text"]:20} '
        f'{entity_label:20} '
        f'{bio_label}'
    )

Sud-Ouest            FOURNISSEUR          B-FOURNISSEUR
ie                   FOURNISSEUR          I-FOURNISSEUR
tige                 FOURNISSEUR          I-FOURNISSEUR
Offre                O                    O
de                   O                    O
prix                 O                    O
n°:                  O                    O
002.143/25           NUMERO_DEVIS         B-NUMERO_DEVIS
Affaire              O                    O
:                    O                    O
A24013               O                    O
CHEVAL               O                    O
Devis                O                    O
effectué             O                    O
par                  O                    O
Olivier              O                    O
CANTELAUBE           O                    O
Beautiran,           O                    O
le                   DATE_DEVIS           B-DATE_DEVIS
28                   DATE_DEVIS           I-DATE_DEVIS
Janvier              DATE_DEVIS           I-DATE_

In [23]:
training_example = {
    "id": "devis_2_page_1",
    "tokens": [word["text"] for word in ocr_words],
    "bboxes": [word["bbox"] for word in ocr_words],
    "ner_tags": bio_labels,
}

In [24]:
print(len(training_example["tokens"]))
print(len(training_example["bboxes"]))
print(len(training_example["ner_tags"]))

173
173
173


In [25]:
assert len(training_example["tokens"]) == len(training_example["bboxes"])
assert len(training_example["tokens"]) == len(training_example["ner_tags"])

In [26]:
for token, bbox, label in zip(
    training_example["tokens"],
    training_example["bboxes"],
    training_example["ner_tags"],
):
    if label != "O":
        print(f"{token:20} {str(bbox):25} {label}")

Sud-Ouest            [113, 69, 196, 87]        B-FOURNISSEUR
ie                   [90, 85, 168, 109]        I-FOURNISSEUR
tige                 [175, 85, 208, 109]       I-FOURNISSEUR
002.143/25           [259, 157, 373, 169]      B-NUMERO_DEVIS
le                   [766, 21, 774, 29]        B-DATE_DEVIS
28                   [779, 22, 793, 29]        I-DATE_DEVIS
Janvier              [796, 21, 842, 29]        I-DATE_DEVIS
2025                 [846, 21, 875, 29]        I-DATE_DEVIS
SFECO                [699, 65, 752, 74]        B-SOCIETE
Lbnbhol              [860, 756, 945, 765]      B-MONTANT_TOTAL


In [27]:
for i, word in enumerate(ocr_words):
    if word["text"] == "bahol":
        print(i)

In [28]:
for i, word in enumerate(ocr_words):
    if word["text"] == "bahol":
        print(f"\n--- Contexte autour du mot {word['text']} ---")

        for j in range(max(0, i - 5), min(len(ocr_words), i + 6)):
            print(
                f"{j:3d} | "
                f"{training_example['tokens'][j]:20} | "
                f"{training_example['ner_tags'][j]}"
            )

In [29]:
montant_region = next(
    r for r in entity_regions
    if r["label"] == "MONTANT_TOTAL"
)

print(montant_region)

{'label': 'MONTANT_TOTAL', 'bbox': [851, 751, 950, 768]}


In [30]:
print(ocr_data["pages"][0]["image_width"])
print(ocr_data["pages"][0]["image_height"])

1191
1684


In [31]:
for word in ocr_words:
    if point_in_bbox(center(word["bbox"]), montant_region["bbox"]):
        print(word)

{'text': 'Lbnbhol', 'confidence': 0.0, 'bbox': [860, 756, 945, 765]}


In [32]:
from datasets import Dataset

/Users/alibahabakar/Projects/document-intelligence-engine/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/alibahabakar/Projects/document-intelligence-engine/.venv/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.4.3)/charset_normalizer (3.4.9) doesn't match a supported version!
  warnings.warn(


In [33]:
training_example = {
    "id": "devis_2_page_1",
    "tokens": [word["text"] for word in ocr_words],
    "bboxes": [word["bbox"] for word in ocr_words],
    "ner_tags": bio_labels,
}

print(training_example.keys())

dataset = Dataset.from_list(
    [training_example]
)

dict_keys(['id', 'tokens', 'bboxes', 'ner_tags'])


In [34]:
print(len(training_example["tokens"]))
print(len(training_example["bboxes"]))
print(len(training_example["ner_tags"]))

173
173
173


In [35]:
from datasets import Dataset

dataset = Dataset.from_list([training_example])

print(dataset)

Dataset({
    features: ['id', 'tokens', 'bboxes', 'ner_tags'],
    num_rows: 1
})


In [36]:
label2id = {
    "O": 0,

    "B-FOURNISSEUR": 1,
    "I-FOURNISSEUR": 2,

    "B-SOCIETE": 3,
    "I-SOCIETE": 4,

    "B-DATE_DEVIS": 5,
    "I-DATE_DEVIS": 6,

    "B-NUMERO_DEVIS": 7,
    "I-NUMERO_DEVIS": 8,

    "B-MONTANT_TOTAL": 9,
    "I-MONTANT_TOTAL": 10,
}

id2label = {
    value: key
    for key, value in label2id.items()
}

print(label2id["B-SOCIETE"])
print(id2label[3])

3
B-SOCIETE


In [37]:
training_example["label_ids"] = [
    label2id[label]
    for label in training_example["ner_tags"]
]

In [38]:
print(training_example["ner_tags"][:10])
print(training_example["label_ids"][:10])

['B-FOURNISSEUR', 'I-FOURNISSEUR', 'I-FOURNISSEUR', 'O', 'O', 'O', 'O', 'B-NUMERO_DEVIS', 'O', 'O']
[1, 2, 2, 0, 0, 0, 0, 7, 0, 0]


In [39]:
dataset = Dataset.from_list([training_example])

print(dataset)

Dataset({
    features: ['id', 'tokens', 'bboxes', 'ner_tags', 'label_ids'],
    num_rows: 1
})


In [40]:
from transformers import LayoutLMv3Processor
processor = LayoutLMv3Processor.from_pretrained(
    "microsoft/layoutlmv3-base",
    apply_ocr=False,
)
print(processor)

LayoutLMv3Processor:
- image_processor: LayoutLMv3ImageProcessor {
  "apply_ocr": false,
  "do_normalize": true,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.5,
    0.5,
    0.5
  ],
  "image_processor_type": "LayoutLMv3ImageProcessor",
  "image_std": [
    0.5,
    0.5,
    0.5
  ],
  "resample": 2,
  "rescale_factor": 0.00392156862745098,
  "size": {
    "height": 224,
    "width": 224
  },
  "tesseract_config": ""
}

- tokenizer: LayoutLMv3Tokenizer(name_or_path='microsoft/layoutlmv3-base', vocab_size=50265, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}, added_tokens_decoder={
	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True)

In [41]:
test_tokens = processor.tokenizer.tokenize("002.143/25")

print(test_tokens)

['Ġ00', '2', '.', '143', '/', '25']


In [42]:
from PIL import Image

image = Image.open(
    "data/label_studio/images/devis_2_page_1.png"
).convert("RGB")

image.size

(1654, 2339)

In [43]:
words = training_example["tokens"][:10]
boxes = training_example["bboxes"][:10]
labels = training_example["label_ids"][:10]

In [44]:
encoding = processor(
    images=image,
    text=words,
    boxes=boxes,
    word_labels=labels,
    truncation=True,
    padding="max_length",
    return_tensors="pt",
)

In [46]:
print(encoding.keys())

for key, value in encoding.items():
    print(key, value.shape)

KeysView({'input_ids': tensor([[    0, 12314,    12,   673,   257,   990, 39420,   326,  7876,  4995,
           241,   263, 13512,  1178,   295, 12938,    35, 16273,   176,     4,
         26332,    73,  1244, 11176,  9556,  4832,     2,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,

In [47]:
token_ids = encoding["input_ids"][0]
token_boxes = encoding["bbox"][0]
token_labels = encoding["labels"][0]
attention_mask = encoding["attention_mask"][0]

tokens = processor.tokenizer.convert_ids_to_tokens(
    token_ids.tolist()
)

for index, (token, box, label_id, attention) in enumerate(
    zip(tokens, token_boxes, token_labels, attention_mask)
):
    # On s'arrête au début du padding.
    if attention.item() == 0:
        break

    label_value = label_id.item()

    label_name = (
        "IGNORED"
        if label_value == -100
        else id2label[label_value]
    )

    print(
        f"{index:>3} | "
        f"{token:<15} | "
        f"{box.tolist()} | "
        f"{label_name}"
    )

  0 | <s>             | [0, 0, 0, 0] | IGNORED
  1 | ĠSud            | [113, 69, 196, 87] | B-FOURNISSEUR
  2 | -               | [113, 69, 196, 87] | IGNORED
  3 | O               | [113, 69, 196, 87] | IGNORED
  4 | u               | [113, 69, 196, 87] | IGNORED
  5 | est             | [113, 69, 196, 87] | IGNORED
  6 | Ġie             | [90, 85, 168, 109] | I-FOURNISSEUR
  7 | Ġt              | [175, 85, 208, 109] | I-FOURNISSEUR
  8 | ige             | [175, 85, 208, 109] | IGNORED
  9 | ĠOff            | [62, 156, 119, 171] | O
 10 | re              | [62, 156, 119, 171] | IGNORED
 11 | Ġde             | [128, 156, 149, 169] | O
 12 | Ġpri            | [157, 156, 207, 172] | O
 13 | x               | [157, 156, 207, 172] | IGNORED
 14 | Ġn              | [215, 157, 251, 169] | O
 15 | Â°              | [215, 157, 251, 169] | IGNORED
 16 | :               | [215, 157, 251, 169] | IGNORED
 17 | Ġ00             | [259, 157, 373, 169] | B-NUMERO_DEVIS
 18 | 2               | [259, 157

In [48]:
from transformers import LayoutLMv3ForTokenClassification
model = LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
)

Loading weights: 100%|██████████| 212/212 [00:00<00:00, 55801.22it/s]
[transformers] LayoutLMv3ForTokenClassification LOAD REPORT from: microsoft/layoutlmv3-base
Key                        | Status  | 
---------------------------+---------+-
classifier.out_proj.bias   | MISSING | 
classifier.out_proj.weight | MISSING | 
classifier.dense.bias      | MISSING | 
classifier.dense.weight    | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [50]:
print(type(model))
print(model.num_parameters())

model.eval()

<class 'transformers.models.layoutlmv3.modeling_layoutlmv3.LayoutLMv3ForTokenClassification'>
125926027


LayoutLMv3ForTokenClassification(
  (layoutlmv3): LayoutLMv3Model(
    (embeddings): LayoutLMv3TextEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (x_position_embeddings): Embedding(1024, 128)
      (y_position_embeddings): Embedding(1024, 128)
      (h_position_embeddings): Embedding(1024, 128)
      (w_position_embeddings): Embedding(1024, 128)
    )
    (patch_embed): LayoutLMv3PatchEmbeddings(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (norm): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)

In [52]:
import torch

with torch.no_grad():
    outputs = model(**encoding)

print(outputs.keys())
print("Loss :", outputs.loss.item())
print("Logits :", outputs.logits.shape)

odict_keys(['loss', 'logits'])
Loss : 2.458177328109741
Logits : torch.Size([1, 512, 11])


In [53]:
print(outputs.logits[0, 1])

tensor([ 0.1325,  0.1073, -0.1233,  0.2187,  0.0997,  0.0179, -0.2671,  0.1511,
        -0.0547,  0.0333,  0.0891])
